# Day 22: Routing, Load Balancing & Queueing
> *Inference Engineering* — Chapter 7.2.3 | Philip Kiely, Baseten Books 2026

**Layer:** Infrastructure | **Prerequisite:** Day 21 (Autoscaling)


## Concept Overview

The load balancer sits in front of GPU workers and decides which worker handles each request. Simple round-robin is insufficient for inference: workers have heterogeneous KV cache state, and load varies per request (prefill length matters). Least-connections routing sends to the worker with the fewest in-flight requests. Cache-aware routing (Day 14) additionally considers prefix cache overlap. Priority queuing separates interactive (latency-sensitive) from batch (throughput-optimized) traffic.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import heapq, time
from collections import deque

print('Load balancing simulation environment')


## 1. Routing Strategies

Compare round-robin, least-connections, and weighted routing.


In [ ]:
class Worker:
    def __init__(self, worker_id, speed=1.0):
        self.id = worker_id
        self.speed = speed
        self.active_requests = 0
        self.total_processed = 0
        self.latencies = []

    def process(self, duration):
        actual_duration = duration / self.speed
        self.active_requests += 1
        # Simulate completion (synchronous for demo)
        self.active_requests -= 1
        self.total_processed += 1
        self.latencies.append(actual_duration)
        return actual_duration

def simulate_routing(strategy, num_workers=4, num_requests=1000, seed=42):
    np.random.seed(seed)
    # Workers with different speeds (heterogeneous)
    workers = [Worker(i, speed=1.0 + i*0.1) for i in range(num_workers)]
    latencies = []
    rr_counter = 0

    request_durations = np.random.exponential(50, num_requests)  # ms

    for i, dur in enumerate(request_durations):
        if strategy == 'round_robin':
            idx = i % num_workers
        elif strategy == 'least_connections':
            idx = min(range(num_workers), key=lambda x: workers[x].active_requests)
        elif strategy == 'random':
            idx = np.random.randint(num_workers)
        elif strategy == 'weighted_round_robin':
            # Route proportionally to worker speed
            weights = [w.speed for w in workers]
            idx = np.random.choice(num_workers, p=np.array(weights)/sum(weights))

        # Simulate the request
        workers[idx].active_requests += 1
        lat = dur / workers[idx].speed
        workers[idx].active_requests -= 1
        workers[idx].total_processed += 1
        latencies.append(lat)

    return {
        'mean_lat': np.mean(latencies),
        'p99_lat': np.percentile(latencies, 99),
        'max_lat': np.max(latencies),
        'worker_loads': [w.total_processed for w in workers],
    }

print(f'{"Strategy":<25} {"Mean (ms)":>10} {"P99 (ms)":>10} {"Max (ms)":>10} {"Load dist":>15}')
print('-' * 75)
for strategy in ['round_robin', 'least_connections', 'random', 'weighted_round_robin']:
    r = simulate_routing(strategy)
    load = ','.join(str(l) for l in r['worker_loads'])
    print(f'{strategy:<25} {r["mean_lat"]:>10.1f} {r["p99_lat"]:>10.1f} {r["max_lat"]:>10.1f} [{load}]')


## 2. Priority Queuing

Separate interactive and batch traffic into different queues to meet latency SLOs for interactive while maximizing throughput for batch.


In [ ]:
class PriorityQueue:
    def __init__(self):
        self._queue = []
        self._count = 0

    def push(self, priority, item):
        # Lower priority number = higher priority
        heapq.heappush(self._queue, (priority, self._count, item))
        self._count += 1

    def pop(self):
        return heapq.heappop(self._queue)[2]

    def __len__(self):
        return len(self._queue)

def simulate_priority_queue(num_requests=500, seed=42):
    np.random.seed(seed)
    pq = PriorityQueue()
    normal_q = deque()

    requests = []
    for i in range(num_requests):
        is_interactive = np.random.random() < 0.2  # 20% interactive
        duration = np.random.exponential(30 if is_interactive else 100)
        arrival = i * 0.5  # requests arrive 0.5 time units apart
        requests.append({'id': i, 'interactive': is_interactive,
                         'duration': duration, 'arrival': arrival})
        if is_interactive:
            pq.push(0, requests[-1])  # high priority
        else:
            pq.push(1, requests[-1])  # low priority

    # Process with priority ordering
    interactive_waits = []
    batch_waits = []
    t = 0
    served = 0
    while len(pq) > 0:
        req = pq.pop()
        wait = max(0, t - req['arrival'])
        if req['interactive']:
            interactive_waits.append(wait)
        else:
            batch_waits.append(wait)
        t += req['duration'] / 100  # normalize
        served += 1

    return interactive_waits, batch_waits

iwait, bwait = simulate_priority_queue()
print(f'Priority Queuing Results:')
print(f'  Interactive requests: {len(iwait)}')
print(f'    Mean wait: {np.mean(iwait):.2f} | P99 wait: {np.percentile(iwait,99):.2f}')
print(f'  Batch requests: {len(bwait)}')
print(f'    Mean wait: {np.mean(bwait):.2f} | P99 wait: {np.percentile(bwait,99):.2f}')
print(f'  Interactive gets {np.mean(bwait)/np.mean(iwait):.1f}x lower wait time')


## 3. Queueing Theory: Sizing for SLOs

M/M/c queue model: arrival rate $\lambda$, service rate $\mu$, $c$ servers. At utilization $\rho = \lambda/(c \cdot \mu)$, wait time $W \propto 1/(1-\rho)$.


In [ ]:
from math import factorial

def erlang_c(c, rho):
    """Erlang C formula: P(wait > 0) for M/M/c queue."""
    if rho >= 1:
        return 1.0
    a = c * rho  # offered load
    numerator = (a**c / factorial(c)) * (c / (c - a))
    denominator = sum(a**k / factorial(k) for k in range(c)) + numerator
    return numerator / denominator

def mean_wait_mmc(lam, mu, c):
    """Mean waiting time in M/M/c queue."""
    rho = lam / (c * mu)
    if rho >= 1:
        return float('inf')
    Pc = erlang_c(c, rho)
    W_queue = Pc / (c * mu * (1 - rho))
    return W_queue * 1000  # ms

# Model an inference cluster
# mu = 1 req/sec per worker (50ms mean service time)
mu = 20.0  # requests per second per worker

print('M/M/c Queueing Analysis: Workers needed for SLO')
print(f'{"Arrival Rate":>15} {"Utilization":>13}', end='')
for c in [1, 2, 4, 8]:
    print(f' {c}w P95<50ms', end='')
print()
print('-' * 75)
for lam in [5, 10, 15, 20, 30, 40, 60]:
    print(f'{lam:>15.0f} rps {lam/(1*mu):>12.0%}', end='')
    for c in [1, 2, 4, 8]:
        w = mean_wait_mmc(lam, mu, c)
        ok = 'YES' if w < 50 else 'NO'
        print(f' {ok:>10}', end='')
    print()


## Experiments: Try These

1. **Power of two choices**: Implement the "power of two choices" algorithm (pick 2 random workers, route to the less loaded). Compare to round-robin and least-connections.
2. **Sticky sessions**: Add session affinity to the load balancer (same user → same worker). Measure KV cache hit rate improvement.
3. **Circuit breaker**: Implement a circuit breaker that stops sending traffic to workers with >5s response times. Test under simulated worker failure.


## Key Takeaways

- Round-robin is wrong for inference: it ignores request duration variability and KV cache state. Least-connections is the minimum viable strategy.
- Priority queuing protects interactive traffic from batch workloads — a 20% interactive / 80% batch mix can give interactive 5-10x better latency.
- Erlang C queueing models give good intuition for worker sizing: at 80% utilization, wait times are 5x higher than at 50% utilization.
- For cache-aware routing, combine least-connections with prefix hash affinity — route to the least-loaded worker that already has the prefix cached.

## References
- *Inference Engineering* Ch 7.2.3 — Philip Kiely, Baseten Books 2026
- Kleinrock, L. (1975), "Queueing Systems, Volume 1"
- NGINX load balancing documentation
